# Data Cleaning 11 -- VIX Futures & CBOE SKEW Combined

## Input
`Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/vix_skew_combined.parquet` (5,286 rows, 2004-01-02 to 2024-12-31, 14 factor columns, keyed on date only)

## Purpose
Cleans the daily macro-level combined VIX futures term structure and CBOE SKEW index data. Key concerns addressed: VIX futures late start (launched March 26, 2004), rolling lookback warmup periods for derived SKEW factors, weekend/holiday date structure, and value range verification.

## Stage 0: Load & Inspect
Basic shape, date range, column inventory, and dtype verification.

## Stage 1: Missing Data Audit
- Per-column NaN counts with explanations for each source of missing data
- VIX futures NaN date verification: confirms all NaN in VIX futures columns fall before the March 26, 2004 launch date, with zero unexpected NaN after launch

## Stage 2: Date Structure
- Day-of-week distribution
- Weekend check
- Date gap distribution and identification of gaps exceeding 5 days (holiday stretches)
- Duplicate date check

## Stage 3: Value Range Checks
Summary statistics (min, median, max, mean) for all 14 factor columns.

## Stage 4: Clean & Save

### No Columns Dropped
All 14 factors retained. NaN rates are all below 2.5% and fully explained by structural causes.

### Structural NaN (All Left As-Is)
- **VIX futures factors** (~60 NaN, 1.1%): VIX futures launched March 26, 2004. The 59 trading days from January 2 to March 25 have no futures data. Same treatment as CFTC starting June 2006 -- structural late start.
- **`vix_fut_volume`** (118 NaN, 2.2%): additional NaN on days with no volume reported, likely early-period low-liquidity days.
- **`skew_pctile_252d`** (131 NaN, 2.5%): 252-day rolling lookback warmup period. First valid value appears approximately January 2005.
- **`skew_chg_5d`, `vix_term_spread_5d_chg`** (~6--11 NaN): 5-day diff lookback.
- **`skew_ma20`, `skew_vs_ma20`** (15 NaN): 20-day moving average warmup.
- **`skew`, `skew_excess`** (6 NaN): holiday gaps.

### No Winsorisation
Applied in the merge pipeline.

### No Forward-Fill Applied Here
This is macro-level daily data where forward-fill is appropriate, but it is deferred to the merge pipeline along with all other macro series (FRED, CFTC, AAII, WRDS macro daily).

## Output
`Data/Data_Collection/Cleaned/11_VIX_SKEW/vix_skew_combined_clean.parquet` -- 14 factor columns (all retained), 5,286 rows

In [1]:
# %% [markdown]
# # Data Cleaning: vix_skew_combined.parquet
#
# Source: Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/vix_skew_combined.parquet
# Output: Data/Data_Collection/Cleaned/11_VIX_SKEW/vix_skew_combined_clean.parquet
#
# Daily macro-level data combining VIX futures term structure (WRDS) and
# CBOE SKEW index. Keyed on date only (no PERMNO). 14 factor columns.

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/11_VIX_Futures_and_CBOE_SKEW/vix_skew_combined.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/11_VIX_SKEW')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — VIX Futures + CBOE SKEW")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

factor_cols = [c for c in df.columns if c != 'date']

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")
print(f"  Factor columns: {len(factor_cols)}")

print(f"\nColumns and dtypes:")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<30s} {str(df[c].dtype):<15s}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Factor':<30s} {'NaN':>6s}  {'%':>6s}  {'Explanation'}")
print("  " + "-" * 75)
for col in col_nan_sorted.index:
    n = int(col_nan[col])
    pct = col_nan_pct[col]
    # Explain each NaN source
    if 'vix_fut' in col or 'vix_term' in col:
        reason = "VIX futures launched 2004-03-26 (Jan-Mar missing)"
    elif col == 'skew_pctile_252d':
        reason = "252-day lookback warmup period"
    elif col in ['skew_chg_5d', 'vix_term_spread_5d_chg']:
        reason = "5-day diff lookback"
    elif col in ['skew_ma20', 'skew_vs_ma20']:
        reason = "20-day MA warmup"
    elif col in ['skew', 'skew_excess']:
        reason = "Holiday gaps"
    else:
        reason = ""
    if n > 0:
        print(f"  {col:<30s} {n:>6d}  {pct:>5.2f}%  {reason}")

# Verify: are VIX futures NaN all before March 26 2004?
print(f"\n--- VIX futures NaN date check ---")
vix_nan_dates = df[df['vix_fut_front'].isna()]['date']
if len(vix_nan_dates) > 0:
    print(f"  VIX futures NaN dates: {vix_nan_dates.min().date()} → {vix_nan_dates.max().date()}")
    n_before_launch = (vix_nan_dates < '2004-03-26').sum()
    n_after_launch = (vix_nan_dates >= '2004-03-26').sum()
    print(f"  Before 2004-03-26 (expected): {n_before_launch}")
    print(f"  After 2004-03-26 (unexpected): {n_after_launch}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: DATE STRUCTURE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: DATE STRUCTURE")
print("=" * 90)

# Day-of-week
print(f"\n--- Day-of-week distribution ---")
dow = df['date'].dt.day_name().value_counts()
print(dow.to_string())

has_weekends = (df['date'].dt.dayofweek >= 5).any()
print(f"\n  Contains weekends: {'YES — need to drop' if has_weekends else 'NO'}")

# Gap analysis
print(f"\n--- Date gap distribution ---")
date_diffs = df['date'].diff().dt.days.dropna()
print(f"  Mean: {date_diffs.mean():.1f}, Median: {date_diffs.median():.0f}")
print(f"  Min: {date_diffs.min():.0f}, Max: {date_diffs.max():.0f}")
print(f"\n  Gap distribution:")
for gap, count in date_diffs.value_counts().sort_index().head(10).items():
    print(f"    {int(gap):>3d} days: {count:>5,d}")

long_gaps = date_diffs[date_diffs > 5]
if len(long_gaps) > 0:
    print(f"\n  Gaps > 5 days: {len(long_gaps)}")
    for idx in long_gaps.sort_values(ascending=False).head(5).index:
        gap_end = df.loc[idx, 'date']
        gap_start = df.loc[idx - 1, 'date']
        print(f"    {gap_start.date()} → {gap_end.date()} ({int(date_diffs.loc[idx])} days)")
else:
    print(f"\n  ✓ No gaps > 5 days")

# Duplicates
n_dupes = df['date'].duplicated().sum()
print(f"\n  Duplicate dates: {n_dupes}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: VALUE RANGE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: VALUE RANGE CHECKS")
print("=" * 90)

print(f"\n  {'Factor':<30s} {'min':>10s}  {'median':>10s}  {'max':>10s}  {'mean':>10s}")
print("  " + "-" * 75)
for col in factor_cols:
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    print(f"  {col:<30s} {vals.min():>10.3f}  {vals.median():>10.3f}  "
          f"{vals.max():>10.3f}  {vals.mean():>10.3f}")

# %% [markdown]
# ## Stage 4: Clean & Save
#
# **Data overview:**
# Daily macro-level data combining VIX futures term structure and CBOE SKEW
# index. 5,286 rows from 2004-01-02 to 2024-12-31. Keyed on date only.
#
# **No columns dropped.** All 14 factors retained. NaN rates are all <2.5%
# and fully explained by structural causes.
#
# **Structural NaN (all left as-is):**
# - VIX futures factors (~60 NaN, 1.1%): VIX futures launched March 26, 2004.
#   The 59 trading days from January 2 to March 25 have no futures data.
#   This is identical to CFTC starting June 2006 — structural late start.
# - `vix_fut_volume` (118 NaN, 2.2%): additional NaN on days with no volume
#   reported, likely early-period low-liquidity days.
# - `skew_pctile_252d` (131 NaN, 2.5%): 252-day rolling lookback warmup.
#   First valid value appears ~January 2005 (one year after sample start).
# - `skew_chg_5d`, `vix_term_spread_5d_chg` (~6–11 NaN): 5-day diff lookback.
# - `skew_ma20`, `skew_vs_ma20` (15 NaN): 20-day MA warmup.
# - `skew`, `skew_excess` (6 NaN): holiday gaps.
#
# **No winsorisation.** Applied in merge pipeline.
#
# **No forward-fill applied here.** This is macro-level daily data where
# forward-fill IS appropriate, but it's deferred to the merge pipeline
# along with all other macro series (FRED, CFTC, AAII, WRDS macro daily).
#
# **Factors retained: 14** (all kept)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 4: CLEAN & SAVE")
print("=" * 90)

# ── 4a. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols].isna().sum()
nan_cols = nan_check[nan_check > 0].sort_values(ascending=False)
total_nan = nan_cols.sum()
total_cells = len(df) * len(factor_cols)

print(f"\n  Total NaN: {total_nan:,} / {total_cells:,} ({total_nan/total_cells*100:.2f}%)")
print(f"  Factors with any NaN: {len(nan_cols)} / {len(factor_cols)}")

# ── 4b. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list:")
for i, c in enumerate(factor_cols, 1):
    vals = df[c].dropna()
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n} NaN, {nan_n/len(df)*100:.1f}%)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<30s} [{vals.min():.3f}, {vals.max():.3f}]{nan_str}")

print(f"\n  Sample (first 5 rows):")
print(df.head(5).to_string(index=False))

# ── 4c. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'vix_skew_combined_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns ({len(factor_cols)} factors)")

print("\nCleaning complete.")

STAGE 0: LOAD & INSPECT — VIX Futures + CBOE SKEW

  Shape: 5,286 rows × 15 columns
  Date range: 2004-01-02 → 2024-12-31
  Unique dates: 5,286
  Factor columns: 14

Columns and dtypes:
    1. vix_fut_front                  Float64        
    2. vix_fut_volume                 Float64        
    3. vix_fut_oi                     Float64        
    4. vix_fut_second                 Float64        
    5. vix_term_spread                Float64        
    6. vix_term_ratio                 Float64        
    7. vix_fut_ret_1d                 Float64        
    8. vix_term_spread_5d_chg         Float64        
    9. skew                           float64        
   10. skew_excess                    float64        
   11. skew_pctile_252d               float64        
   12. skew_chg_5d                    float64        
   13. skew_ma20                      float64        
   14. skew_vs_ma20                   float64        

STAGE 1: MISSING DATA AUDIT

--- Per-Column NaN ---

  Fa